In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import kruskal, shapiro

from scipy.stats import mannwhitneyu
from itertools import combinations
from statsmodels.stats.multitest import multipletests

import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
import seaborn as sns

/tmp/ipykernel_249118/477489196.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.4)
  from scipy import stats


## Sanity check and RMA scale projection

In [2]:
bipole_evaluation = pd.read_csv('ProjectionValidation/BipoleEvaluation.csv')
effect_size = pd.read_csv('ProjectionValidation/EffectSize.csv')

In [3]:
rma_proj = np.round(bipole_evaluation[(bipole_evaluation['embedding'] == 'SBERT') & (bipole_evaluation['scale'] != 'sanity_check')].groupby(['scale', 'label'])[['score']].mean(), 4)

# bipole_evaluation[bipole_evaluation['scale'] != 'sanity_check'].groupby(by = ['embedding', 'scale', 'label'])[['score']].mean()

In [4]:
results = []
for scale in bipole_evaluation['scale'].unique():
    if scale == 'sanity_check':
        continue
    grp      = bipole_evaluation[(bipole_evaluation['embedding'] == 'SBERT') & (bipole_evaluation['scale'] == scale)]
    myth     = grp[grp['label'] == 'myth']['score'].dropna().values
    debunked = grp[grp['label'] == 'debunked']['score'].dropna().values
    diff     = myth - debunked
    n        = len(diff)
    _, p_norm = shapiro(diff)
    if p_norm > 0.05:
        print('normal')
        from scipy.stats import ttest_rel
        stat, p = ttest_rel(myth, debunked)
        test = "paired_t"
    else:
        print('non normal')
        stat, p = wilcoxon(diff)
        test = "wilcoxon"
    mean_d   = diff.mean()
    sd_d     = diff.std(ddof=1)
    dz       = mean_d / sd_d
    cf       = 1 - (3 / (4 * (n - 1) - 1))
    results.append({'scale': scale, 'test': test, 'stat': stat, 'p': p,
                    'n': n, 'mean_myth': myth.mean(), 'mean_debunked': debunked.mean(),
                    'cohens_dz': dz, 'hedges_g': dz * cf})

print(np.round(pd.DataFrame(results)[['scale', 'n', 'mean_myth', 'mean_debunked', 'hedges_g']], 3))

normal
normal
   scale   n  mean_myth  mean_debunked  hedges_g
0   IRMA  40      0.171         -0.244     3.870
1  AMMSA  30      0.117         -0.203     2.019


## Original Narratives Projection

In [5]:
df = pd.read_csv('ProjectionValidation/NarrativeProjections-withDemographics.csv')

In [6]:
df.columns

Index(['narrative_index', 'projection_sbert', 'projection_w2v',
       'projection_glove', 'adult_victim', 'childhood_abuse',
       'perpetrator_female', 'perpetrator_male', 'victim_female',
       'victim_male', 'first_person_perpetrator', 'first_person_victim',
       'third_person_perpetrator', 'third_person_victim',
       'acquaintance_assault', 'family_member', 'intimate_partner',
       'stranger_assault', 'clothing', 'perpetrator_intoxication',
       'resistance', 'victim_intoxication'],
      dtype='object')

In [7]:
for col in ['projection_w2v', 'projection_glove', 'projection_sbert']:
    mean, std = np.round(df[col].mean(), 4),  np.round(df[col].std(), 4)
    print(f"{col.split('_')[1]} & {mean} & {std} \\")

w2v & 0.1656 & 0.0508 \
glove & 0.7788 & 0.1522 \
sbert & -0.0975 & 0.076 \


In [8]:
MYTHS = ['clothing', 'perpetrator_intoxication','resistance', 'victim_intoxication']
embeddings = ['projection_sbert', 'projection_w2v', 'projection_glove']

In [110]:
# for myth in MYTHS:
#     print(df.groupby([myth])[['projection_w2v', 'projection_glove', 'projection_sbert']].mean())

In [9]:
# fig, axes = plt.subplots(1, len(MYTHS), figsize=(18, 3))

# for ax, myth in zip(axes, MYTHS):
#     table = pd.DataFrame({
#         emb: df.groupby(myth)[emb].mean()
#         for emb in embeddings
#     }).T
#     table = table[['entailment', 'neutral', 'contradiction']]
#     table.index = [e.replace('projection_', '') for e in embeddings]
#     sns.heatmap(table, annot=True, fmt='.3f', ax=ax, cmap='coolwarm', center=table.values.mean())
#     ax.set_title(myth.replace('_', ' '))
#     ax.set_xlabel('')

# plt.tight_layout()
# # plt.show()
# plt.savefig("Figures/OriginalNarratives-Embeddings.pdf")

In [10]:
# fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# for ax, emb in zip(axes, embeddings):
#     table = pd.DataFrame({
#         myth: df.groupby(myth)[emb].mean()
#         for myth in MYTHS
#     }).T
#     table = table[['entailment', 'neutral', 'contradiction']]
#     sns.heatmap(table, annot=True, fmt='.3f', ax=ax)
#     ax.set_title(emb.replace('projection_', ''))
#     ax.set_xlabel('')

# plt.tight_layout()
# # plt.show()
# plt.savefig("Figures/OriginalNarratives-Myths.pdf")

In [ ]:
# df.groupby(['clothing'])[['projection_w2v', 'projection_glove', 'projection_sbert']].mean()

In [53]:
# for name, nli_feat in [("MYTHS", MYTHS), ("AGE", AGE), ("GENDER", GENDER), ("PERSPECTIVE", PERSPECTIVE), ("RELATIONSHIP", RELATIONSHIP)]:
#     all_neutral_mask = (df[nli_feat] == 'neutral').all(axis=1)
#     vals = df.loc[all_neutral_mask, 'projection_sbert']
#     print(f"{name}: n = {len(vals)}, mean = {vals.mean():.3f}, std = {vals.std():.3f}")

## Original: Stratified Statistical Testing

In [5]:
# narr1 = pd.read_csv('ProjectionValidation/NarrativeProjections.csv')

In [6]:
# narr2 = pd.read_csv('ProjectionValidation/NarrativeProjections-withDemographics.csv')

In [36]:
# (narr1[['narrative_index', 'projection_sbert', 'projection_w2v','projection_glove']] == narr2[['narrative_index', 'projection_sbert', 'projection_w2v','projection_glove']]).all()

In [12]:
GROUPS = {
    'Age': ['adult_victim', 'childhood_abuse'],
    'Gender': ['perpetrator_female', 'perpetrator_male', 'victim_female', 'victim_male'],
    'Perspective': ['first_person_perpetrator', 'first_person_victim', 'third_person_perpetrator', 'third_person_victim'],
    'Relationship': ['acquaintance_assault', 'family_member', 'intimate_partner', 'stranger_assault'],
    'Myths': ['clothing', 'perpetrator_intoxication', 'resistance', 'victim_intoxication']
}
features = [f for group in GROUPS.values() for f in group]
nli_labels = ['entailment', 'neutral', 'contradiction']

AGE = ['adult_victim', 'childhood_abuse']
GENDER = ['perpetrator_female', 'perpetrator_male', 'victim_female','victim_male']
PERSPECTIVE = ['first_person_perpetrator', 'first_person_victim','third_person_perpetrator', 'third_person_victim']
RELATIONSHIP = ['acquaintance_assault', 'family_member', 'intimate_partner','stranger_assault']
MYTHS = ['clothing', 'perpetrator_intoxication','resistance', 'victim_intoxication']


In [13]:
# Shapiro-Wilk with Bonferroni correct for multiple tests
results = []
for feat in features:
    for label in df[feat].unique():
        vals = df.loc[df[feat] == label, 'projection_sbert'].dropna()
        if len(vals) < 3:
            continue
        stat, p = stats.shapiro(vals.sample(min(len(vals), 5000), random_state=42))
        results.append({'feature': feat, 'label': label, 'n': len(vals),
                        'mean': round(vals.mean(), 3), 'std': round(vals.std(), 3),
                        'skewness': round(vals.skew(), 3), 'kurtosis': round(vals.kurtosis(), 3),
                        'shapiro_p': p})
shapiro_df = pd.DataFrame(results)

In [14]:
shapiro_df['non_normal_dist'], shapiro_df['p_corrected'], _, alpha = multipletests(shapiro_df['shapiro_p'], method='fdr_bh')

In [17]:
np.round(alpha * 10**4, 2)

np.float64(9.26)

In [18]:
pivot = shapiro_df.pivot(index='feature', columns='label', values=['n', 'p_corrected', 'non_normal_dist'])

In [319]:
# shapiro_df

In [321]:
def format_cell(row, label):
    n = row['n'][label]
    non_normal_dist = row['non_normal_dist'][label]
    return f'\\textbf{{{n}}}' if non_normal_dist else str(n)

latex_rows = []
for feat, row in pivot.iterrows():
    feat = " ".join(feat.split('_'))
    cells = [feat] + [format_cell(row, label) for label in ['entailment', 'neutral', 'contradiction']]
    latex_rows.append(' & '.join(cells) + ' \\\\')

# print('\n'.join(latex_rows))

In [323]:
# Kruskal-Wallis
results = []
for feat in features:
    groups = [df.loc[df[feat] == label, 'projection_sbert'].dropna()
              for label in df[feat].unique() if len(df.loc[df[feat] == label]) >= 3]
    if len(groups) < 2:
        continue
    stat, p = kruskal(*groups)
    n, k = sum(len(g) for g in groups), len(groups)
    eta2 = (stat - k + 1) / (n - k)
    results.append({'feature': feat, 'kruskal_stat': round(stat, 3), 'p_value': p, 'eta2': round(eta2, 4)})

kruskal_df = pd.DataFrame(results)
kruskal_df['reject?'], kruskal_df['p_corrected'], _, alpha = multipletests(kruskal_df['p_value'], method='bonferroni')

In [324]:
np.round(alpha * 10**3, 2)

np.float64(2.78)

In [265]:
# kruskal_df

In [333]:
def format_eta(row):
    val = f'{row["eta2"]:.4f}'
    return f'\\textbf{{{val}}}' if row['reject?'] else val

print('\n'.join(
    f'{row["feature"]} & {format_eta(row)} \\\\' 
    for _, row in kruskal_df.iterrows()
))

adult_victim & \textbf{0.0557} \\
childhood_abuse & \textbf{0.0381} \\
perpetrator_female & \textbf{0.0262} \\
perpetrator_male & \textbf{0.0270} \\
victim_female & \textbf{0.0096} \\
victim_male & \textbf{0.0048} \\
first_person_perpetrator & \textbf{0.0154} \\
first_person_victim & 0.0018 \\
third_person_perpetrator & \textbf{0.0297} \\
third_person_victim & \textbf{0.0026} \\
acquaintance_assault & \textbf{0.0070} \\
family_member & \textbf{0.0211} \\
intimate_partner & \textbf{0.0089} \\
stranger_assault & \textbf{0.0154} \\
clothing & 0.0009 \\
perpetrator_intoxication & \textbf{0.0094} \\
resistance & 0.0000 \\
victim_intoxication & \textbf{0.0103} \\


In [269]:
results = []

for feat in features:
    for l1, l2 in list(combinations(nli_labels, 2)):
        g1 = df.loc[df[feat] == l1, 'projection_sbert'].dropna()
        g2 = df.loc[df[feat] == l2, 'projection_sbert'].dropna()
        if len(g1) < 3 or len(g2) < 3:
            continue
        stat, p = mannwhitneyu(g1, g2, alternative='two-sided')
        r = 1 - (2 * stat) / (len(g1) * len(g2))
        results.append({'feature': feat, 'pair': f'{l1} vs {l2}', 'p_value': p, 'r': round(r, 4)})
mwu_df = pd.DataFrame(results)
mwu_df['reject?'], mwu_df['p_corrected'], _, alpha = multipletests(mwu_df['p_value'], method='bonferroni')

In [275]:
pivot = mwu_df.pivot(index='feature', columns='pair', values=['r', 'p_corrected', 'reject?'])

In [342]:
np.round(alpha * 10**3, 2)

np.float64(2.78)

In [338]:
pair_cols = [f'{l1} vs {l2}' for l1, l2 in combinations(nli_labels, 2)]

def format_r(feat, pair):
    row = mwu_df[(mwu_df['feature'] == feat) & (mwu_df['pair'] == pair)]
    r, sig = row['r'].values[0], row['reject?'].values[0]
    val = f'{r:.4f}'
    return f'\\textbf{{{val}}}' if sig else val

print('\n'.join(
    f'{" ".join(feat.split('_'))} & ' + ' & '.join(format_r(feat, pair) for pair in pair_cols) + ' \\\\'
    for feat in features
))

adult victim & 0.0221 & \textbf{-0.2606} & \textbf{-0.2740} \\
childhood abuse & \textbf{0.2376} & \textbf{0.2118} & -0.0228 \\
perpetrator female & 0.0058 & \textbf{-0.2288} & \textbf{-0.2245} \\
perpetrator male & \textbf{0.2165} & \textbf{0.2245} & 0.0072 \\
victim female & \textbf{0.1569} & -0.0820 & \textbf{-0.2309} \\
victim male & \textbf{0.2032} & \textbf{0.1582} & -0.0481 \\
first person perpetrator & 0.1283 & -0.0757 & \textbf{-0.2062} \\
first person victim & \textbf{0.0597} & 0.0316 & -0.0259 \\
third person perpetrator & \textbf{0.2909} & \textbf{0.2133} & -0.0955 \\
third person victim & 0.1048 & -0.0001 & \textbf{-0.1112} \\
acquaintance assault & \textbf{0.1564} & \textbf{0.2232} & 0.0724 \\
family member & \textbf{0.2490} & \textbf{0.1795} & -0.0644 \\
intimate partner & \textbf{-0.1138} & \textbf{-0.2274} & \textbf{-0.1123} \\
stranger assault & -0.2008 & -0.3597 & \textbf{-0.1473} \\
clothing & 0.0653 & 0.1012 & 0.0327 \\
perpetrator intoxication & \textbf{-0.1353} &

In [334]:
pair_cols

['entailment vs neutral',
 'entailment vs contradiction',
 'neutral vs contradiction']

In [18]:
# features = [
#     'adult_victim', 'childhood_abuse', 'perpetrator_female', 'perpetrator_male',
#     'victim_female', 'victim_male', 'first_person_perpetrator', 'first_person_victim',
#     'third_person_perpetrator', 'third_person_victim', 'acquaintance_assault',
#     'family_member', 'intimate_partner', 'stranger_assault'
# ]
# embeddings = ['projection_sbert']
# # 'projection_w2v', 'projection_glove']
# nli_order = ['entailment', 'neutral', 'contradiction']

# def melt_for_feature(df, feature):
#     melted = df[[feature] + embeddings].melt(id_vars=feature, var_name='embedding', value_name='projection')
#     melted.rename(columns={feature: 'nli'}, inplace=True)
#     melted['nli'] = pd.Categorical(melted['nli'], categories=nli_order, ordered=True)
#     return melted

# fig, axes = plt.subplots(len(features), 1, figsize=(10, len(features) * 4))

# for ax, feat in zip(axes, features):
#     melted = melt_for_feature(narr_df, feat)
#     sns.violinplot(data=melted, x='nli', y='projection', hue='embedding', ax=ax,
#                    split=False, inner='quartile', density_norm='width')
#     ax.set_title(feat)
#     ax.set_xlabel('')
#     ax.legend(fontsize=7, loc='upper right')
    
# plt.tight_layout()
# plt.show()

In [17]:
# palette = {'entailment': '#4C72B0', 'neutral': '#CCB974', 'contradiction': '#C44E52'}

# fig, axes = plt.subplots(len(features), 1, figsize=(10, len(features) * 4))
# for ax, feat in zip(axes, features):
#     melted = melt_for_feature(narr_df, feat)
#     sns.violinplot(data=melted, x='embedding', y='projection', hue='nli', ax=ax,
#                    split=False, inner='quartile', density_norm='width', palette=palette,
#                    hue_order=nli_order)
#     ax.set_title(feat)
#     ax.set_xlabel('')
#     ax.legend(fontsize=7, loc='upper right')
# plt.tight_layout()
# plt.show()

In [21]:
# features = [
#     'adult_victim', 'childhood_abuse', 'perpetrator_female', 'perpetrator_male',
#     'victim_female', 'victim_male', 'first_person_perpetrator', 'first_person_victim',
#     'third_person_perpetrator', 'third_person_victim', 'acquaintance_assault',
#     'family_member', 'intimate_partner', 'stranger_assault'
# ]
# nli_labels = ['entailment', 'neutral', 'contradiction']

# index = pd.MultiIndex.from_product([features, nli_labels], names=['feature', 'label'])
# cross = pd.DataFrame(index=index, columns=index, dtype=object)

# for f1 in features:
#     for l1 in nli_labels:
#         mask1 = df[f1] == l1
#         for f2 in features:
#             for l2 in nli_labels:
#                 vals = df.loc[mask1 & (df[f2] == l2), 'projection_sbert']
#                 if len(vals) == 0:
#                     cross.loc[(f1, l1), (f2, l2)] = '-'
#                 else:
#                     cross.loc[(f1, l1), (f2, l2)] = f'{vals.mean():.3f} ± {vals.std():.3f}'

# cross

In [61]:
results = []
for feat in features:
    for label in nli_labels:
        vals = df.loc[df[feat] == label, 'projection_sbert']
        results.append({
            'feature': feat,
            'label': label,
            'n': len(vals),
            'mean': f'{vals.mean():.3f}',
            'std': f'{vals.std():.3f}'
        })

# pd.DataFrame(results).pivot(index='feature', columns='label', values=['mean', 'std'])

NameError: name 'features' is not defined